# 03c — Alberta foothills: candidate protection areas

Crops **and masks** the aligned stack to (Alberta boundary ∩ eastern-foothills polygon) and runs
with **compactness off** (all penalties 0) so small high-value areas can surface. Parameters live
in `config.ANALYSES["ab_foothills"]`; outputs → `output_data/iter6_ab_foothills/`.

**Prerequisite:** requires `input_data/alberta_boundary/alberta.gpkg` and
`input_data/ab_foothills/foothills.gpkg`. Until they exist, cell 1's manifest refresh stops the
run by design. **Kernel:** `R (y2y)`. Ethan runs cell-by-cell.

In [13]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "ab_foothills"   # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=ab_foothills)
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_foothills | solver=highs (single solution)
objective=min_shortfall | budget=45% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=intersect bounds=[-1504000, 1164000, -953000, 2055000] (+mask) | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_ab_foothills


In [14]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ROI intersect: cropped to 551 x 891 cells + polygon mask
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 551 x 891 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [15]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

planning units: 76,057 cells | budget = 45% = 34,226 cells
locked-in [pa_mask]: 23,123 cells (30.4% of window) -- fits within budget


In [16]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)


In [17]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

penalties -> connectivity=0 | boundary=0 | neighbor=0  (0 = off)


In [18]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

Warning message in problem(x, zones(features, zone_names = names(x), feature_names = names(features)), :
“→ `features` has a layer with only zero values.”
A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (76057 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -1504000, 1164000, -953000, 2055000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 34225.65)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (23123 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200

In [19]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

LP has 49 rows; 76105 cols; 1057330 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [3e-02, 1e+00]

  Bound   [1e+00, 1e+00]

  RHS     [3e+04, 1e+05]


Presolving model

36 rows, 52969 cols, 724992 nonzeros 0s

0 rows, 0 cols, 0 nonzeros 0s

Presolve reductions: rows 0(-49); columns 0(-76105); nonzeros 0(-1057330) - Reduced to empty

Performed postsolve

Solving the original LP from the solution after postsolve



Model status        : Optimal

Objective value     :  4.0271710118e+00

P-D objective error :  9.8094198054e-15

HiGHS run time      :          0.35

solved with highs: 1 solution(s) in 0.7 s


In [20]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01   34225.65         45             11102


In [21]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter6_ab_foothills/portfolio.tif
  output_data/iter6_ab_foothills/selection_frequency.tif
  output_data/iter6_ab_foothills/portfolio_representation.csv
  output_data/iter6_ab_foothills/run_summary.json
